# 🔗 LangChain Core Concepts — LCEL and Runnables

## Learning Objectives
In this notebook, you will learn:
1. **LCEL (LangChain Expression Language)** - How to compose prompts, models, and parsers into a pipeline with the `|` operator
2. **The Runnable Interface** - The common `invoke` / `batch` / `stream` methods every LCEL chain exposes
3. **Batch and Streaming Execution** - How to run a chain over multiple inputs at once and receive tokens as they're generated
4. **Schema Inspection** - How to introspect a chain's expected input and output shapes
5. **Provider-Agnostic Initialization** - Using `init_chat_model` versus instantiating a provider class (`ChatOpenAI`, `ChatAnthropic`) directly

## Prerequisites
- Basic understanding of Python functions
- An `OPENAI_API_KEY` set in a `.env` file at the project root (loaded via `python-dotenv`)
- Familiarity with the concept of prompt templates

> Converted from `01_core_concepts.py` - part of **01 LangChain Foundations**.

---
## 📦 Part 1: Environment Setup

This section loads environment variables from `.env` (API keys) and imports the LangChain building blocks used throughout the notebook: prompt templates, output parsers, and both the provider-specific and provider-agnostic ways to create a chat model.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Configuration
# ============================================================================
from dotenv import load_dotenv

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model

load_dotenv()

print("✅ Environment variables loaded from .env")

---
## ⛓️ Part 2: LCEL and the Runnable Interface

**LCEL (LangChain Expression Language)** lets you compose a prompt, a model, and an output parser into a single pipeline using the `|` operator. Every LCEL chain implements the shared `Runnable` interface, so the resulting chain object exposes `invoke`, `batch`, `stream`, and schema-inspection methods regardless of what's inside it.

### Key Concepts:
- **Runnable**: The shared interface (`invoke`, `batch`, `stream`, `ainvoke`, ...) implemented by prompts, models, parsers, and any chain built by piping them together
- **`|` operator**: Composes two runnables into a new runnable that pipes the first's output into the second's input

### 2.1 🔗 Basic Chain

`demo_basic_chain` composes a prompt, a chat model, and an output parser into a single chain with the pipe operator, then invokes it with one input.

In [ ]:
# ============================================================================
# DEMO_BASIC_CHAIN: Compose and Invoke a Basic LCEL Chain
# ============================================================================
def demo_basic_chain():
    """Demonstrates a basic chain using LCEL and Runnables."""

    # Component 1: Define the prompt template using LCEL
    prompt = ChatPromptTemplate.from_template(
        "You are a helpful assistant. Answer in one sentence: {question}"
    )
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    parser = StrOutputParser()

    # Compose with pipe operator
    chain = prompt | model | parser

    # Execute the chain with an input
    result = chain.invoke({"question": "What is LangChain?"})
    print(f"Response: {result}")

    return chain

### 2.2 📦 Batch Execution

`demo_batch_exectution` runs the same chain over a list of multiple inputs at once via `.batch()`, which is more efficient than looping and calling `.invoke()` for each input.

In [ ]:
# ============================================================================
# DEMO_BATCH_EXECTUTION: Run a Chain Over Multiple Inputs at Once
# ============================================================================
def demo_batch_exectution():
    """Demonstrate batch execution for multiple inputs."""
    prompt = ChatPromptTemplate.from_template("Translate to French: {text}")
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    parser = StrOutputParser()

    chain = prompt | model | parser

    # Batch - run with multiple inputs
    inputs = [
        {"text": "Hello, how are you?"},
        {"text": "What is your name?"},
        {"text": "Where is the nearest restaurant?"},
    ]
    results = chain.batch(inputs)

    for text in zip(inputs, results):
        print(f"Input: {text[0]['text']} => Output: {text[1]}")

### 2.3 🌊 Streaming

`demo_streaming` shows how to consume a chain's output token-by-token as it is generated, using `.stream()` instead of waiting for the full `.invoke()` result.

In [ ]:
# ============================================================================
# DEMO_STREAMING: Consume Chain Output Token-by-Token
# ============================================================================
def demo_streaming():
    """Demonstrate streaming for real-time output."""
    prompt = ChatPromptTemplate.from_template("Write a haiku about: {topic}")
    model = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.7,
    )
    parser = StrOutputParser()

    chain = prompt | model | parser

    # Streaming - run with streaming enabled
    print("Streaming output: ")
    for chunk in chain.stream({"topic": "nature"}):
        print(chunk, end="", flush=True)
    print()  # for newline after streaming

### 2.4 🔍 Schema Inspection

`demo_schema_inspection` shows how to introspect a chain's `input_schema` and `output_schema` as JSON schema — useful for validating what a chain expects and returns before wiring it into an application.

In [ ]:
# ============================================================================
# DEMO_SCHEMA_INSPECTION: Inspect a Chain's Input/Output Schemas
# ============================================================================
def demo_schema_inspection():
    """Demonstrate input/output schema inspection."""
    prompt = ChatPromptTemplate.from_template("Summarize the following text: {text}")
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    parser = StrOutputParser()

    chain = prompt | model | parser

    # Inspect input and output schemas
    input_schema = chain.input_schema.model_json_schema()
    output_schema = chain.output_schema.model_json_schema()

    print(f"Input Schema: {input_schema}")
    print(f"Output Schema: {output_schema}")

---
## ✏️ Part 3: Exercise — Build Your Own Chain

`exercise_first_chain` is a worked exercise: build a chain that takes a product name and target audience, generates a marketing tagline with an LLM, and returns it as a plain string.

In [ ]:
# ============================================================================
# EXERCISE_FIRST_CHAIN: Build a Marketing Tagline Chain
# ============================================================================
# Exercise: Build your first chain
def exercise_first_chain():
    """
    EXERCISE: Create a chain that:
    1. Takes a product name and target audience
    2. Generates a marketing tagline
    3. Returns just the tagline as a string

    Test with: product="AI Course", audience="developers"
    """

    # YOUR CODE HERE
    prompt = ChatPromptTemplate.from_template(
        "Create a marketing tagline for a product named '{product}' targeting '{audience}'."
    )
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    parser = StrOutputParser()

    chain = prompt | model | parser

    # Test the chain
    result = chain.invoke({"product": "AI Course", "audience": "developers"})
    print(f"Marketing Tagline: {result}")

---
## 🆕 Part 4: Provider-Agnostic Model Initialization

`init_chat_model` is LangChain's universal, provider-agnostic way to create a chat model from a model name string, instead of importing and instantiating a provider-specific class like `ChatOpenAI` or `ChatAnthropic` directly. `new_way` contrasts both approaches.

In [ ]:
# ============================================================================
# NEW_WAY: Universal vs. Provider-Specific Model Initialization
# ============================================================================
def new_way():
    # the univeral way to initialize a model
    model = init_chat_model("gpt-4o-mini", temperature=0.7, max_tokens=1500)

    # Or provider-specific (still works)

    from langchain_openai import ChatOpenAI
    from langchain_anthropic import ChatAnthropic

    openai_model = ChatOpenAI(model="gpt-4o-mini",
                              temperature=0.7,
                              max_tokens=1500,
                              timeout=30,
                              max_retries=3)

    # anthropic_model = ChatAnthropic(model="claude-sonnet-4-5-20250929")

---
## ▶️ Part 5: Running the Demos

Jupyter sets `__name__` to `"__main__"`, so the original `__main__` guard from the source `.py` file runs as-is in this cell. Uncomment whichever demo you want to run.

In [ ]:
# ============================================================================
# RUN: Execute the Selected Demo
# ============================================================================
if __name__ == "__main__":
    # demo_basic_chain()
    demo_batch_exectution()
    # demo_streaming()
    # demo_schema_inspection()
    # exercise_first_chain()

---
## 📝 Summary

In this notebook, we learned:

### 1. LCEL and the Runnable Interface
- **`|` operator**: Composes a prompt, model, and parser into a single chain
- **`invoke()`**: Runs a chain on a single input
- **`batch()`**: Runs a chain on a list of inputs efficiently
- **`stream()`**: Yields chain output incrementally as it is generated
- **`input_schema` / `output_schema`**: Introspect what a chain expects and returns

### 2. Model Initialization
- **`init_chat_model()`**: A provider-agnostic way to create a chat model from a model name string
- Provider-specific classes like `ChatOpenAI` and `ChatAnthropic` still work and expose the same `Runnable` interface

Functions defined in this notebook:
- `demo_basic_chain()`
- `demo_batch_exectution()`
- `demo_streaming()`
- `demo_schema_inspection()`
- `exercise_first_chain()`
- `new_way()`

### Next Steps
- Continue to `02_working_with_llms.ipynb` to go deeper on working with LLMs in LangChain